# Zero-, One- und Few-Shot Prompting

## Lernziel

Du erstellst drei Prompts für **dieselbe Aufgabe** und vergleichst anschließend ihre Ergebnisse:

1. Zero-Shot: kein Beispiel
2. One-Shot: ein Beispiel
3. Few-Shot: drei Beispiele

Du veränderst nur die Beispiele im Prompt. Die Ausführung und Bewertung sind vorgegeben.


## 0 · Setup

Führe die nächsten beiden Zellen aus. Das Notebook nutzt das in `helfer.py` konfigurierte Modell und zwei getrennte Datensätze: Beispiele für die Prompts und unbekannte Testfälle für die Bewertung.


In [ ]:
import json
import sys
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                  Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

from helfer import BASIS_URL, MODELL, frage_llm, lade_daten

TESTFAELLE = lade_daten("02_cve_testfaelle")
BEISPIELE = lade_daten("02_shot_beispiele")

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")
print(f"{len(BEISPIELE)} Beispiele, {len(TESTFAELLE)} Testfälle")


In [ ]:
print("Testantwort:", frage_llm("Reply with exactly: ready"))


## 1 · Die Aufgabe des Modells

Das Modell erhält eine englische CVE-Kurzbeschreibung und soll **genau ein JSON-Objekt** erzeugen:

```json
{"severity": "high", "component": "Beispielprodukt", "action": "patch"}
```

Dabei gelten feste Werte:

| Feld | Erlaubte Werte |
|---|---|
| `severity` | `critical`, `high`, `medium`, `low` |
| `component` | Produktname aus der Beschreibung |
| `action` | `upgrade`, `patch`, `rotate_credentials`, `disable_feature`, `isolate`, `monitor` |

Die Bewertungsfunktion ist bereits fertig. Sie prüft, ob gültiges JSON zurückkommt und ob jedes Feld dem Sollwert entspricht. Die Testfälle werden **nicht** als Beispiele verwendet.


In [ ]:
FELDER = ["severity", "component", "action"]


def lies_json(antwort):
    try:
        daten = json.loads(antwort)
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(daten, dict) or not all(feld in daten for feld in FELDER):
        return None
    return daten


def bewerte(antworten, sollwerte):
    gelesen = [lies_json(antwort) for antwort in antworten]
    ergebnis = {
        "JSON": sum(daten is not None for daten in gelesen) / len(sollwerte)
    }
    for feld in FELDER:
        treffer = sum(
            daten is not None
            and str(daten[feld]).strip().lower() == str(soll[feld]).strip().lower()
            for daten, soll in zip(gelesen, sollwerte)
        )
        ergebnis[feld] = treffer / len(sollwerte)
    ergebnis["gesamt"] = sum(
        daten is not None
        and all(
            str(daten[feld]).strip().lower() == str(soll[feld]).strip().lower()
            for feld in FELDER
        )
        for daten, soll in zip(gelesen, sollwerte)
    ) / len(sollwerte)
    return ergebnis


def formatiere_beispiel(beispiel):
    output = json.dumps({feld: beispiel[feld] for feld in FELDER})
    return f"Input:\n{beispiel['text']}\nOutput:\n{output}"


def teste_prompt_bauer(bauer, beispiele):
    marker = "EINDEUTIGE-TEST-EINGABE"
    prompt = bauer(marker)
    assert isinstance(prompt, str), "Der Prompt muss ein String sein."
    assert marker in prompt, "Die aktuelle Eingabe fehlt im Prompt."
    for beispiel in beispiele:
        assert beispiel["text"] in prompt, f"Beispiel {beispiel['id']} fehlt."
        assert json.dumps({feld: beispiel[feld] for feld in FELDER}) in prompt, \
            f"Der erwartete Output von {beispiel['id']} fehlt."
    fremde = [b for b in BEISPIELE if b not in beispiele]
    assert all(b["text"] not in prompt for b in fremde), \
        "Der Prompt enthält mehr Beispiele als vorgesehen."
    return prompt


print("Bewertung und Hilfsfunktionen sind bereit.")


## 2 · Vorgegebene Beispiele

Für One-Shot verwendest du das erste Beispiel. Für Few-Shot verwendest du die drei gezeigten Beispiele. Du musst die Beispiele nicht erfinden oder auswählen — deine Aufgabe ist nur, sie sinnvoll in den Prompt einzubauen.


In [ ]:
ONE_SHOT_BEISPIEL = BEISPIELE[0]
FEW_SHOT_BEISPIELE = BEISPIELE[:3]

for beispiel in FEW_SHOT_BEISPIELE:
    print(formatiere_beispiel(beispiel))
    print("─" * 80)


## Challenge 1 · Zero-Shot

Implementiere `baue_zero_shot(eingabe)`.

Der Prompt soll enthalten:

- die Aufgabe,
- das verlangte JSON-Schema und die erlaubten Werte,
- die aktuelle `eingabe`,
- **kein Beispiel**.

Formuliere direkt und eindeutig. Du erstellst nur den Prompt; die Modellaufrufe kommen später.


In [ ]:
def baue_zero_shot(eingabe):
    # TODO: Erstelle den Zero-Shot-Prompt.
    # Er enthält die Aufgabe und die Eingabe, aber kein Beispiel.
    pass


In [ ]:
prompt_zero = teste_prompt_bauer(baue_zero_shot, [])
print("✅ Zero-Shot-Prompt ist vollständig.\n")
print(prompt_zero)


## Challenge 2 · One-Shot

Implementiere `baue_one_shot(eingabe)`.

Nutze dieselbe Aufgabe wie beim Zero-Shot-Prompt und ergänze **genau ein vollständiges Input-/Output-Beispiel**: `ONE_SHOT_BEISPIEL`.

Wichtig: Ein Beispiel besteht immer aus der Beschreibung **und** dem dazugehörigen JSON. Danach folgt die aktuelle Eingabe.


In [ ]:
def baue_one_shot(eingabe):
    # TODO: Erstelle den One-Shot-Prompt.
    # Nutze ONE_SHOT_BEISPIEL als vollständiges Input-/Output-Paar.
    pass


In [ ]:
prompt_one = teste_prompt_bauer(baue_one_shot, [ONE_SHOT_BEISPIEL])
print("✅ One-Shot-Prompt enthält genau ein Beispiel.\n")
print(prompt_one)


## Challenge 3 · Few-Shot

Implementiere `baue_few_shot(eingabe)`.

Nutze dieselbe Aufgabe und ergänze **alle drei** Einträge aus `FEW_SHOT_BEISPIELE` als vollständige Input-/Output-Paare. Danach folgt wieder die aktuelle Eingabe.

Tipp: Formatiere jedes Beispiel mit `formatiere_beispiel(...)` und verbinde die Blöcke mit Leerzeilen.


In [ ]:
def baue_few_shot(eingabe):
    # TODO: Erstelle den Few-Shot-Prompt.
    # Nutze alle drei Einträge aus FEW_SHOT_BEISPIELE als Input-/Output-Paare.
    pass


In [ ]:
prompt_few = teste_prompt_bauer(baue_few_shot, FEW_SHOT_BEISPIELE)
print("✅ Few-Shot-Prompt enthält genau drei Beispiele.\n")
print(prompt_few)


## 3 · Alle Varianten ausführen

Jetzt werden alle drei Prompts an denselben Testfällen gemessen. Dadurch ist der Vergleich fair. Je nach Rechner und Modell kann der Lauf einige Minuten dauern.


In [ ]:
def fuehre_aus(name, bauer):
    antworten = []
    for nummer, fall in enumerate(TESTFAELLE, start=1):
        antworten.append(frage_llm(bauer(fall["text"]), max_tokens=150))
        print(f"\r{name}: {nummer}/{len(TESTFAELLE)}", end="")
    print()
    return antworten


ANTWORTEN = {
    "Zero-Shot": fuehre_aus("Zero-Shot", baue_zero_shot),
    "One-Shot": fuehre_aus("One-Shot", baue_one_shot),
    "Few-Shot": fuehre_aus("Few-Shot", baue_few_shot),
}


In [ ]:
ERGEBNISSE = {
    name: bewerte(antworten, TESTFAELLE)
    for name, antworten in ANTWORTEN.items()
}

spalten = ["JSON", "severity", "component", "action", "gesamt"]
kopf = f"{'Variante':<14}" + "".join(f"{spalte:>12}" for spalte in spalten)
print(kopf)
print("─" * len(kopf))
for name, ergebnis in ERGEBNISSE.items():
    print(f"{name:<14}" + "".join(f"{ergebnis[s]:>12.0%}" for s in spalten))


## 4 · Offene Diskussion

Betrachtet eure Tabelle und besprecht:

1. Welche Variante erzielt bei eurem Modell das beste Gesamtergebnis?
2. Wo unterscheiden sich die Varianten: beim JSON-Format oder bei einzelnen Inhalten?
3. Hat ein Beispiel geholfen? Haben drei Beispiele zusätzlich geholfen?
4. Falls Zero-Shot genauso gut oder besser ist: Welche möglichen Erklärungen gibt es?
5. Welche Aussage könnt ihr aus diesem Lauf treffen — und welche noch nicht?

Es gibt kein vorgegebenes Gewinnergebnis. Das Resultat hängt unter anderem vom Modell, den Beispielen und den Testfällen ab. Entscheidend ist, dass ihr die Wirkung von Beispielen **messt**, statt sie vorauszusetzen.
